## **Implementação de uma rede neural LSTM**

### *Objetivo:* 

Aplicar conceitos de Redes Neurais Recorrentes do tipo LSTM no desenvolvimento de um modelo de *deep learning* para dados sequenciais, como séries temporais, dados financeiros, sensores, consumo, clima, tráfego, texto ou outro conjunto adequado. Preparar o *dataset*, construir janelas temporais quando necessário. Treinar uma LSTM base e realizar um ajuste de hiperparâmetros. Analisar o impacto das escolhas no desempenho do modelo.

### *1. Escolha e Preparação do Dataset*

O dataset que escolhemos para realizar esta atividade foi o [Metro Interstate Traffic Volume](https://archive.ics.uci.edu/dataset/492/metro+interstate+traffic+volume) da *UC Irvine Machine Learning Repository*. A escolha proveio do fato do conjunto de dados já relativamente conhecido para a construção de LSTMs, por se tratar de uma série temporal com uma variável alvo clara. 

A UCI informa que o dataset contém volume horário de tráfego na rodovia I-94, em Minneapolis–St Paul, entre 2012 e 2018, incluindo clima e feriados. A variável alvo é `traffic_volume`; as entradas podem ser histórico de tráfego, temperatura, chuva, neve, nuvens, descrição do clima, hora, dia da semana ou feriado. Logo, vamos tentar prever o tráfego de um intervalo de 24 horas a partir do conjunto de dados fornecidos.




In [52]:
pip install ucimlrepo

In [53]:
# importando o dataset
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
metro_interstate_traffic_volume = fetch_ucirepo(id=492) 
  
# data (as pandas dataframes) 
X = metro_interstate_traffic_volume.data.features 
y = metro_interstate_traffic_volume.data.targets 
  
# metadata 
print(metro_interstate_traffic_volume.metadata) 
  
# variable information 
print(metro_interstate_traffic_volume.variables) 

{'uci_id': 492, 'name': 'Metro Interstate Traffic Volume', 'repository_url': 'https://archive.ics.uci.edu/dataset/492/metro+interstate+traffic+volume', 'data_url': 'https://archive.ics.uci.edu/static/public/492/data.csv', 'abstract': 'Hourly Minneapolis-St Paul, MN traffic volume for westbound I-94. Includes weather and holiday features from 2012-2018.', 'area': 'Other', 'tasks': ['Regression'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 48204, 'num_features': 8, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['traffic_volume'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2019, 'last_updated': 'Fri Mar 15 2024', 'dataset_doi': '10.24432/C5X60B', 'creators': ['John Hogue'], 'intro_paper': None, 'additional_info': {'summary': 'Hourly Interstate 94 Westbound traffic volume for MN DoT ATR station 301, roughly midway between Minneapolis and St Paul, MN. Hourly weath

*1.1 Juntando os dados em um só dataframe*

In [54]:
import pandas as pd
import numpy as np

df = X.copy()
df["traffic_volume"] = y

df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


*1.2 Analisando variáveis*

In [55]:
df.shape

(48204, 9)

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   holiday              61 non-null     object 
 1   temp                 48204 non-null  float64
 2   rain_1h              48204 non-null  float64
 3   snow_1h              48204 non-null  float64
 4   clouds_all           48204 non-null  int64  
 5   weather_main         48204 non-null  object 
 6   weather_description  48204 non-null  object 
 7   date_time            48204 non-null  object 
 8   traffic_volume       48204 non-null  int64  
dtypes: float64(3), int64(2), object(4)
memory usage: 3.3+ MB


In [57]:
df.describe()

,temp,rain_1h,snow_1h,clouds_all,traffic_volume
count,48204.000000,48204.000000,48204.000000,48204.000000,48204.000000
mean,281.205870,0.334264,0.000222,49.362231,3259.818355
std,13.338232,44.789133,0.008168,39.015750,1986.860670
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,272.160000,0.000000,0.000000,1.000000,1193.000000
50%,282.450000,0.000000,0.000000,64.000000,3380.000000
75%,291.806000,0.000000,0.000000,90.000000,4933.000000
max,310.070000,9831.300000,0.510000,100.000000,7280.000000


À primeira vista, não há valores ausentes no dataset, o que é ótimo. Mas já vejo que date_time está como tipo `object`, ou seja, texto puro, logo, vamos convertê-la para podermos analisar a sequência temporal. Além disso, apesar de parecer ter muitos valores ausentes na coluna `holiday`, na verdade ela possui o nome dos feriados e `NaN` quando um dia não é feriado, mas vamos transformar isso em 0 para dias comuns e 1 para feriados para simplificar.

*1.3 Convertendo colunas*

In [58]:
df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

In [59]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [60]:
print(df["date_time"].min(), df["date_time"].max())


2012-10-02 09:00:00 2018-09-30 23:00:00


Intervalo temporal está correto, confirmando que organizamos o dataframe de acordo com a data e convertemos a coluna corretamente.

In [61]:
df["holiday"] = df["holiday"].astype("string")

holiday_series = pd.Series(df["holiday"])

for index, item in holiday_series.items():
    if pd.isna(item):
        holiday_series[index] = '0'
    elif type(item) is type("string"):
        holiday_series[index] = '1'

holiday_series.value_counts()
## Tipicamente é dito que não é uma boa prática iterar por séries do pandas e que se você está fazendo isso,
## provavelmente está fazendo algo errado. Mas, não consegui pensar em outro jeito de fazer que não fosse trabalhoso.
## Poderia ter feito um map com o nome de cada feriado, mas seria chato demais (┬┬﹏┬┬)

,count
holiday,
0,48143
1,61


In [62]:
df["holiday"] = holiday_series.astype(int)
df.rename(columns={"holiday": "is_holiday"}, inplace=True)
df

,is_holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,0,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,0,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,0,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,0,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,0,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918
...,...,...,...,...,...,...,...,...,...
48199,0,283.45,0.0,0.0,75,Clouds,broken clouds,2018-09-30 19:00:00,3543
48200,0,282.76,0.0,0.0,90,Clouds,overcast clouds,2018-09-30 20:00:00,2781
48201,0,282.73,0.0,0.0,90,Thunderstorm,proximity thunderstorm,2018-09-30 21:00:00,2159
48202,0,282.09,0.0,0.0,90,Clouds,overcast clouds,2018-09-30 22:00:00,1450


In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   is_holiday           48204 non-null  int64         
 1   temp                 48204 non-null  float64       
 2   rain_1h              48204 non-null  float64       
 3   snow_1h              48204 non-null  float64       
 4   clouds_all           48204 non-null  int64         
 5   weather_main         48204 non-null  object        
 6   weather_description  48204 non-null  object        
 7   date_time            48204 non-null  datetime64[ns]
 8   traffic_volume       48204 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(3), object(2)
memory usage: 3.3+ MB


Transformação bem-sucedida!

Para finalizar o tratamento de dados, vamos verificar se não há duplicatas.

*1.4 Tratamento de duplicatas*

In [64]:
print(df.duplicated().sum(), "\n", df.isna().sum())

17 
 is_holiday             0
temp                   0
rain_1h                0
snow_1h                0
clouds_all             0
weather_main           0
weather_description    0
date_time              0
traffic_volume         0
dtype: int64


In [65]:
df.loc[df.duplicated()]

,is_holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
18697,0,286.290,0.0,0.0,1,Clear,sky is clear,2015-09-30 19:00:00,3679
23851,0,289.060,0.0,0.0,90,Clouds,overcast clouds,2016-06-01 10:00:00,4831
26784,0,289.775,0.0,0.0,56,Clouds,broken clouds,2016-09-21 15:00:00,5365
26980,0,287.860,0.0,0.0,0,Clear,Sky is Clear,2016-09-29 19:00:00,3435
27171,0,279.287,0.0,0.0,56,Clouds,broken clouds,2016-10-07 18:00:00,4642
28879,0,267.890,0.0,0.0,90,Snow,light snow,2016-12-06 18:00:00,4520
29268,0,254.220,0.0,0.0,1,Clear,sky is clear,2016-12-19 00:00:00,420
34711,0,295.010,0.0,0.0,40,Clouds,scattered clouds,2017-06-21 11:00:00,4808
34967,0,292.840,0.0,0.0,1,Clear,sky is clear,2017-06-30 10:00:00,4638
34969,0,294.520,0.0,0.0,1,Clear,sky is clear,2017-06-30 11:00:00,4725


In [66]:
df.loc[df["date_time"] == pd.to_datetime("2018-09-29 19:00:00")]

,is_holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
48172,0,280.68,0.0,0.0,90,Clouds,overcast clouds,2018-09-29 19:00:00,3818
48173,0,280.68,0.0,0.0,90,Clouds,overcast clouds,2018-09-29 19:00:00,3818


In [67]:
## Aparentemente existem alguns duplicatas, então vou simplesmente removê-los já que não há sentido para uma análise temporal mantê-los.

df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

Antes de criar as janelas da LSTM, ainda falta um detalhe importante: verificar se h? mais de uma linha para o mesmo hor?rio. Em s?ries temporais, isso atrapalha porque a rede espera uma sequ?ncia ordenada, com um registro por passo de tempo.

In [ ]:
df["date_time"].duplicated().sum()

Como existem hor?rios repetidos, vou manter uma ?nica linha por `date_time`. Para vari?veis num?ricas, uso a m?dia; para clima em texto, uso a moda; para feriado, mantenho 1 se qualquer linha daquele hor?rio for feriado.

In [ ]:
def moda_primeiro_valor(serie):
    return serie.mode().iloc[0]


df = (
    df.sort_values("date_time")
    .groupby("date_time", as_index=False)
    .agg({
        "is_holiday": "max",
        "temp": "mean",
        "rain_1h": "mean",
        "snow_1h": "mean",
        "clouds_all": "mean",
        "weather_main": moda_primeiro_valor,
        "weather_description": moda_primeiro_valor,
        "traffic_volume": "mean",
    })
)

df["traffic_volume"] = df["traffic_volume"].round().astype(int)
df.shape

In [ ]:
df["date_time"].diff().value_counts().head()

Agora a base est? pronta para virar entrada da LSTM: sem valores ausentes, sem duplicatas exatas, sem hor?rios repetidos e com vari?veis temporais que podem representar ciclos.

Já que o dataframe utiliza o tipo `datetime` do pandas, precisamos quebrar o tempo em colunas separadas para que a rede seja capaz de entender separadamente os padrões de hora, dia da semana, mês, etc. 

In [68]:
df["year"] = df["date_time"].dt.year
df["month"] = df["date_time"].dt.month
df["day"] = df["date_time"].dt.day
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek  # segunda=0, domingo=6
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

df.head()

,is_holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,year,month,day,hour,day_of_week,is_weekend
0,0,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,2012,10,2,9,1,0
1,0,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,2012,10,2,10,1,0
2,0,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,2012,10,2,11,1,0
3,0,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,2012,10,2,12,1,0
4,0,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,2012,10,2,13,1,0


No notebook de exemplo de LSTM que vimos duranta a aula, as métricas de preço são normalizadas, o que funciona, mas aqui não podemos simplesmente normalizar por se tratar de uma escala temporal. Por exemplo, os dias são cíclicos, 23h é próximo de 00h, mas caso façamos a normalização, a rede vai entender que 23h é muito longe de 00h. Portanto, precisamos usar uma escala **cíclica**, ou seja, utilizamos seno e cosseno, já que variam entre 0 e 1.

In [69]:
## 2πr = comprimento da circunferência de raio cuja hora vai definir
## / 24 = para definir o intervalo de 24h
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["days_of_week_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["days_of_week_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df.head()

,is_holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,year,...,day,hour,day_of_week,is_weekend,hour_sin,hour_cos,days_of_week_sin,days_of_week_cos,month_sin,month_cos
0,0,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,2012,...,2,9,1,0,7.071068e-01,-0.707107,0.781831,0.62349,-0.866025,0.5
1,0,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,2012,...,2,10,1,0,5.000000e-01,-0.866025,0.781831,0.62349,-0.866025,0.5
2,0,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,2012,...,2,11,1,0,2.588190e-01,-0.965926,0.781831,0.62349,-0.866025,0.5
3,0,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,2012,...,2,12,1,0,1.224647e-16,-1.000000,0.781831,0.62349,-0.866025,0.5
4,0,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,2012,...,2,13,1,0,-2.588190e-01,-0.965926,0.781831,0.62349,-0.866025,0.5
